## 검증 코드 통일 후 공통 Fold 생성

### 1. Google Drive와 Train 구조 확인

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
import ast
import hashlib
import platform
import sys

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

drive_project = Path('/content/drive/MyDrive/SNU')
local_project = Path.cwd()
project_candidates = [drive_project, local_project, local_project.parent]
PROJECT_DIR = next(
    (path for path in project_candidates if (path / 'data' / 'train.csv').is_file()),
    None,
)
assert PROJECT_DIR is not None, (
    'data/train.csv를 찾지 못했습니다. Google Drive의 SNU 폴더 구조를 확인하세요.'
)

DATA_DIR = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'splits'

SEED = 42
N_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3

train = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig')
required_columns = {
    'Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4', 'Sentence', 'Answer'
}
assert required_columns <= set(train.columns), train.columns.tolist()
assert len(train) == 9_535, f'예상 Train 9,535개와 다릅니다: {len(train):,}'
assert train['Id'].is_unique, 'Train Id가 중복됩니다.'

def parse_answer(value: str) -> list[int]:
    answer = ast.literal_eval(value)
    assert isinstance(answer, list)
    assert len(answer) == 4
    assert sorted(answer) == [1, 2, 3, 4]
    return answer

train['answer_tuple'] = train['Answer'].apply(lambda x: str(parse_answer(x)))
assert train['answer_tuple'].nunique() == 24

print(f'프로젝트: {PROJECT_DIR}')
print(f'Train samples: {len(train):,}')
print(f'Answer classes: {train["answer_tuple"].nunique()}')
print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')
print(f'networkx: {nx.__version__}')
print(f'scikit-learn: {sklearn.__version__}')

### 2. Train 이미지 38,140장의 MD5 계산

각 Train 샘플의 Input 1~4 이미지 byte가 완전히 동일한지 확인하기 위해 MD5를 계산합니다.

이미지 내용을 분석하거나 특징을 추출하는 단계가 아니라 exact duplicate를 찾는 단계입니다. Google Drive 속도에 따라 수 분 정도 걸릴 수 있습니다.

In [ ]:
# Step 1: 이미지 해시 계산
def compute_md5(filepath):
    with open(filepath, 'rb') as file:
        return hashlib.md5(file.read()).hexdigest()

def get_all_image_records(dataframe, image_dir):
    records = []
    for _, row in dataframe.iterrows():
        for input_number in range(1, 5):
            path = image_dir / row['Id'] / row[f'Input_{input_number}']
            records.append({'sample_id': row['Id'], 'path': str(path)})
    return records

train_records = get_all_image_records(train, TRAIN_IMAGE_DIR)
assert len(train_records) == 38_140

paths = [record['path'] for record in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(
        tqdm(
            executor.map(compute_md5, paths),
            total=len(paths),
            desc='Train image MD5',
        )
    )

for record, image_hash in zip(train_records, hashes):
    record['hash'] = image_hash

assert all('hash' in record for record in train_records)
print(f'MD5 계산 완료: {len(train_records):,}장')
print(f'고유 image hash: {len(set(hashes)):,}개')

### 3. 공유 이미지 기반 sample component 생성

한 image hash가 최대 3개 샘플에서 나타날 때만 그룹 연결 근거로 사용합니다. 4개 이상 샘플에서 반복되는 hash는 검은 화면이나 범용 재사용 프레임처럼 서로 무관한 영상을 과도하게 연결할 수 있어 제외합니다. 동일한 eligible hash를 공유하는 샘플은 graph edge로 연결되고, 연결된 component 전체가 하나의 `cluster_id`를 받습니다.

In [ ]:
# Step 2: 3개 초과 반복 등장 이미지는 제외하고 그래프로 클러스터링
hash_counter = defaultdict(set)
for record in train_records:
    hash_counter[record['hash']].add(record['sample_id'])

hash_to_samples = {
    image_hash: sample_ids
    for image_hash, sample_ids in hash_counter.items()
    if len(sample_ids) <= MAX_SAMPLES_PER_HASH
}
excluded_hash_count = sum(
    len(sample_ids) > MAX_SAMPLES_PER_HASH
    for sample_ids in hash_counter.values()
)

graph = nx.Graph()
graph.add_nodes_from(train['Id'].unique())
for _, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for first_index in range(len(sample_ids)):
        for second_index in range(first_index + 1, len(sample_ids)):
            graph.add_edge(
                sample_ids[first_index],
                sample_ids[second_index],
            )

components = list(nx.connected_components(graph))
id_to_cluster = {
    sample_id: cluster_id
    for cluster_id, component in enumerate(components)
    for sample_id in component
}
train['cluster_id'] = train['Id'].map(id_to_cluster)

assert train['cluster_id'].notna().all()
assert train['cluster_id'].nunique() == len(components)
component_sizes = pd.Series([len(component) for component in components])

print(f'그룹 연결에서 제외된 frequent hash: {excluded_hash_count:,}개')
print(f'총 component: {len(components):,}개')
print(f'최대 component 크기: {component_sizes.max():,}')
print('큰 component 상위 10개:')
display(component_sizes.sort_values(ascending=False).head(10).to_frame('size'))

### 4. StratifiedGroupKFold로 Fold 0~4 배정

`cluster_id`를 group으로, 24개 `Answer` 순열을 stratification label로 사용합니다.

같은 component는 반드시 한 Fold 안에만 들어가며, 각 Fold가 전체 Answer 분포를 가능한 한 비슷하게 갖도록 배정합니다. 이 셀이 Sinyoung 검증 코드의 핵심 분할 단계입니다.

In [ ]:
# Step 3: 그룹 단위 Stratified K-Fold 분할
sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)
train['fold'] = -1

for fold, (_, validation_indices) in enumerate(
    sgkf.split(
        train,
        train['answer_tuple'],
        groups=train['cluster_id'],
    )
):
    train.loc[validation_indices, 'fold'] = fold

assert set(train['fold'].unique()) == set(range(N_FOLDS))
print('Fold 배정 완료')

### 5. Fold 무결성 및 label 균형 검사

같은 component가 여러 Fold로 갈라지지 않았는지, 모든 ID가 정확히 한 Fold에 배정됐는지, 각 Fold에 24개 Answer class가 모두 있는지 검사합니다.

Fold 개수는 약 1,900개씩이어야 하지만 특정 숫자를 강제로 요구하지는 않습니다. 실행 환경에 따른 차이를 확인한 뒤 이 실행에서 생성된 CSV 하나를 팀 공통 파일로 고정합니다.

In [ ]:
# Step 4: split 검증
cluster_fold_counts = train.groupby('cluster_id')['fold'].nunique()
cross_fold_cluster_count = int((cluster_fold_counts > 1).sum())
assert cross_fold_cluster_count == 0, (
    '같은 component가 여러 Fold에 걸쳐 있습니다.'
)
assert len(train) == 9_535
assert train['Id'].nunique() == 9_535
assert not (train['fold'] == -1).any()

fold_counts = train['fold'].value_counts().sort_index()
fold_class_counts = train.groupby('fold')['answer_tuple'].nunique().sort_index()
assert (fold_class_counts == 24).all(), fold_class_counts.to_dict()

fold_balance = (
    train.groupby('fold')['answer_tuple']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
mean_class_std = float(fold_balance.std(axis=0).mean())

print(f'Cross-fold component: {cross_fold_cluster_count}')
print(f'Fold 간 Answer 비율 표준편차 평균: {mean_class_std:.6f}')
display(pd.DataFrame({
    'sample_count': fold_counts,
    'answer_class_count': fold_class_counts,
}))
print(f'Fold 0 Train: {(train["fold"] != 0).sum():,}')
print(f'Fold 0 Validation: {(train["fold"] == 0).sum():,}')

### 6. 공통 `fold_assignments.csv` 저장

각 Train ID와 Fold 번호만 저장합니다. 이후 모든 팀원은 K-Fold를 다시 생성하지 않고 이 CSV를 `train.csv`와 `Id` 기준으로 merge해 사용합니다.

저장 후 다시 읽어 ID와 Fold가 완전히 동일한지 검사하고 SHA-256 checksum을 출력합니다. 팀원들이 같은 checksum을 확인하면 정확히 같은 파일을 사용하는 것입니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLD_ASSIGNMENTS_CSV = OUTPUT_DIR / 'fold_assignments.csv'

fold_assignments = train[['Id', 'fold']].copy()
fold_assignments['fold'] = fold_assignments['fold'].astype(int)
fold_assignments.to_csv(
    FOLD_ASSIGNMENTS_CSV,
    index=False,
    encoding='utf-8-sig',
)

saved_assignments = pd.read_csv(
    FOLD_ASSIGNMENTS_CSV,
    dtype={'Id': str, 'fold': int},
    encoding='utf-8-sig',
)
pd.testing.assert_frame_equal(
    fold_assignments.reset_index(drop=True),
    saved_assignments.reset_index(drop=True),
)

with FOLD_ASSIGNMENTS_CSV.open('rb') as file:
    file_sha256 = hashlib.sha256(file.read()).hexdigest()

print(f'저장 완료: {FOLD_ASSIGNMENTS_CSV}')
print(f'저장 행: {len(saved_assignments):,}')
print(f'SHA-256: {file_sha256}')
display(saved_assignments.head())

### 7. Fold 0 학습·검증 사용 방식 확인

Fold 0만 Validation으로 사용합니다. `fold != 0`인 Fold 1~4는 학습 데이터이고, `fold == 0`은 검증 데이터입니다. 이 셀은 ID 집합만 확인하며 모델을 학습하지 않습니다.

In [ ]:
merged_train = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig').merge(
    saved_assignments,
    on='Id',
    how='left',
    validate='one_to_one',
)
assert merged_train['fold'].notna().all()

fold0_train = merged_train[merged_train['fold'] != 0].copy()
fold0_validation = merged_train[merged_train['fold'] == 0].copy()
assert set(fold0_train['Id']).isdisjoint(set(fold0_validation['Id']))
assert len(fold0_train) + len(fold0_validation) == len(merged_train)

print(f'Fold 0 학습 샘플: {len(fold0_train):,}')
print(f'Fold 0 검증 샘플: {len(fold0_validation):,}')
print('검증 split 생성 완료: 다음 단계는 새 Qwen3 LoRA 학습 코드 작성입니다.')